# Synapse: The RAG Iteration Lab

Welcome to the capstone lab for **Course 5: RAG Systems**. 

In the previous sessions, you've built the individual parts of a RAG pipeline: retrieval quality, metadata filtering, and grounded answering. In this lab, you stop adding parts and start **iterating on the whole**.

### 🎯 The Real-World Scenario
You are an AI Engineer at **SynapseCloud**, a SaaS company. You've built a support-docs RAG assistant to help users with billing, data retention, and export policies. 

**The Problem:** The assistant "feels" okay in demos, but users are reporting hallucinations on complex policy questions. Your manager has asked: *"Is it actually any good, and how do we know if we're making it better?"*

### 🧠 The Engineering Narrative
By the end of this notebook, you will have moved from **vibe-based tweaking** to a **measured iteration loop**:
1.  **The Naive Tension:** Building a loop that works on easy cases but fails on a "Gold Set."
2.  **The Runtime Conflict:** Realizing that "fixing" accuracy by adding more context destroys your latency and cost budgets.
3.  **The AI-Native Breakthrough:** Using the **RAG Eval Row** and **Structured Outputs** to surgically locate and fix failures.

## 🛠️ Step 0: Setup

We'll use the OpenAI Python SDK and Pydantic for our structured evaluation artifacts.

In [ ]:
import os, json, time
from typing import List, Optional
from pydantic import BaseModel, Field
from openai import OpenAI
import numpy as np

client = OpenAI()
print("✅ Environment Ready")

## 📖 Step 1: The Corpus & The Naive Loop

Here is our "SynapseCloud" policy corpus. It contains overlapping and plan-specific information.

In [ ]:
corpus = [
    {"id": "doc_1", "text": "Billing Cycle: SynapseCloud offers Monthly and Yearly billing. Yearly billing offers a 20% discount.", "metadata": {"plan": "all"}},
    {"id": "doc_2", "text": "Data Retention (Standard): Data is retained for 30 days after account deletion.", "metadata": {"plan": "standard"}},
    {"id": "doc_3", "text": "Data Retention (Pro): Data is retained for 365 days after account deletion for audit compliance.", "metadata": {"plan": "pro"}},
    {"id": "doc_4", "text": "Log Export: Only Pro plan users can export raw audit logs via the Security Dashboard.", "metadata": {"plan": "pro"}},
    {"id": "doc_5", "text": "Upgrade Policy: Upgrades to Pro are instant. Pro features (like Log Export) are enabled immediately.", "metadata": {"plan": "all"}}
]

def get_embedding(text: str):
    return client.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding

# Pre-calculate embeddings for our corpus
for doc in corpus:
    doc["embedding"] = get_embedding(doc["text"])

### The Naive RAG Function

This is a standard RAG loop: 
1. Embed the query. 
2. Find the top-K chunks by cosine similarity.
3. Stuff them into the prompt.

In [ ]:
def naive_retrieve(query: str, k=2):
    qe = np.array(get_embedding(query))
    scores = []
    for doc in corpus:
        de = np.array(doc["embedding"])
        scores.append((doc, np.dot(qe, de)))
    
    # Sort by score descending and take top-K
    sorted_docs = sorted(scores, key=lambda x: x[1], reverse=True)
    return [d[0] for d in sorted_docs[:k]]

def naive_rag(query: str, k=2):
    start_time = time.time()
    context_docs = naive_retrieve(query, k=k)
    context_text = "\n".join([f"[{d['id']}] {d['text']}" for d in context_docs])
    
    prompt = f"""Use the following context to answer the user query. 
If the answer isn't in the context, say you don't know.

Context:
{context_text}

Query: {query}"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return {
        "answer": response.choices[0].message.content,
        "latency": time.time() - start_time,
        "tokens": response.usage.total_tokens
    }

# Test it
print(naive_rag("How long is my data kept on the Pro plan?")["answer"])

## 📉 Step 2: The Systematic Failure (The Gold Set)

Demos look great, but production is about **edge cases**. We'll now run a "Gold Set" of 3 queries and establish a baseline.

In [ ]:
gold_set = [
    {
        "query": "How long is data kept for Pro users?",
        "expected_doc": "doc_3"
    },
    {
        "query": "Can I export logs on the standard plan?",
        "expected_doc": "doc_4" # doc_4 says 'Only Pro plan users', so it's the evidence needed for a 'No'
    },
    {
        "query": "I just upgraded to Pro. Can I export logs now and how long is my data kept?",
        "expected_docs": ["doc_3", "doc_4", "doc_5"]
    }
]

print("Running Gold Set...")
for entry in gold_set:
    res = naive_rag(entry["query"], k=2)
    print(f"\nQuery: {entry['query']}")
    print(f"Answer: {res['answer']}")
    print(f"Latency: {res['latency']:.2f}s | Tokens: {res['tokens']}")

### 🔎 The Problem:
Look at the results for the last query. It probably missed one of the critical docs (either data retention or export policy) because we only retrieved **Top-2**. 

The naive fix is usually: **"Just increase K!"**

In [ ]:
# Trying to fix it the 'brute force' way
res_expensive = naive_rag(gold_set[2]["query"], k=5)
print(f"\nExpensive Fix (k=5) Latency: {res_expensive['latency']:.2f}s")
print(f"Expensive Fix (k=5) Tokens: {res_expensive['tokens']}")

## 🛠️ Step 3: The Brittle Boundary (Regex Citations)

To know if an answer is grounded, we need **citations**. Before Structured Outputs, engineers had to parse text with Regex to find `[doc_id]` patterns.

In [ ]:
import re

def verify_citations_regex(text):
    citations = re.findall(r"\[(doc_\d+)\]", text)
    return list(set(citations))

sample_answer = "Your data is kept for 365 days [doc_3]."
print(f"Regex Citations: {verify_citations_regex(sample_answer)}")

### 🔍 Research Step: The Industry Moves Fast

Before we move to the breakthrough, perform a quick search. 

**Search Query:** `OpenAI Python SDK Structured Outputs Pydantic example` 

Look for the `.parse()` method. How does it differ from the standard `.create()` method when handling response schemas?

## 🚀 Step 4: The AI-Native Breakthrough (Structured Outputs & Eval Rows)

Instead of guessing, we use the **RAG Eval Row** artifact to make the system measurable, and **Structured Outputs** to make citations guaranteed.

In [ ]:
class GroundedAnswer(BaseModel):
    answer: str = Field(description="The answer to the user query.")
    citations: List[str] = Field(description="List of document IDs used to support the answer.")
    is_supported: bool = Field(description="Whether the answer is fully supported by the provided context.")

class RAGEvalRow(BaseModel):
    query: str
    retrieved_ids: List[str]
    answer: str
    citations: List[str]
    latency: float
    tokens: int
    cost: float
    
def measured_rag(query: str, k=2):
    start_time = time.time()
    
    # 1. Retrieval Stage
    docs = naive_retrieve(query, k=k)
    retrieved_ids = [d["id"] for d in docs]
    context_text = "\n".join([f"ID: {d['id']}\nContent: {d['text']}" for d in docs])
    
    # 2. Generation Stage (Structured Output)
    completion = client.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {"role": "system", "content": "You are a support assistant. Answer queries using ONLY the provided context."},
            {"role": "user", "content": f"Context:\n{context_text}\n\nQuery: {query}"}
        ],
        response_format=GroundedAnswer
    )
    
    res = completion.choices[0].message.parsed
    
    # 3. Calculate Cost (Approximate gpt-4o-2024-08-06 pricing)
    # $2.50 / 1M input, $10.00 / 1M output tokens
    usage = completion.usage
    cost = (usage.prompt_tokens * 2.5e-6) + (usage.completion_tokens * 10e-6)
    
    return RAGEvalRow(
        query=query,
        retrieved_ids=retrieved_ids,
        answer=res.answer,
        citations=res.citations,
        latency=time.time() - start_time,
        tokens=usage.total_tokens,
        cost=cost
    )

# Run the evaluation
row = measured_rag(gold_set[2]["query"], k=2)
print("--- RAG EVAL ROW ---")
print(row.model_dump_json(indent=2))

### 🏆 The Breakthrough: Stage-Specific Debugging

Look at your `retrieved_ids` vs. the `gold_set` expected docs. 

If `doc_4` was missing from `retrieved_ids`, you don't change the prompt. You change the **Retrieval Stage** (e.g., Hybrid Search or Metadata Filtering).

If `doc_4` was retrieved but NOT in `citations`, you don't change retrieval. You change the **Prompt Stage** or the **Model Choice**.

### 🏁 Final Challenge
Modify the `measured_rag` function or the `k` parameter to achieve:
1.  **100% Citation Accuracy** (all expected docs found for the upgrade query).
2.  **Latency < 1.5s**.
3.  **Cost < $0.005**.

Can you do it without just "retrieving everything"?